# Bài 5a — Germany

Trang PDF 45

**Sinh viên:** NGUYỄN LỘC — **MSSV:** 6451071043

**Lưu ý:** Notebook tự chạy độc lập sau khi có dataset. Khi dùng Colab, cài `mlxtend`, `openpyxl` và các thư viện nếu cần.

In [ ]:
!pip -q install mlxtend openpyxl seaborn scikit-learn beautifulsoup4

In [ ]:
from pathlib import Path
import os, re, io, zipfile, time, warnings, string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from sklearn.preprocessing import MinMaxScaler
from bs4 import BeautifulSoup
warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid')
HO_TEN = 'NGUYỄN LỘC'
MSSV = '6451071043'
BASE = Path('/content/lab1_data') if Path('/content').exists() else Path('/mnt/data')
BASE.mkdir(parents=True, exist_ok=True)
OUTPUT = BASE / 'results'; OUTPUT.mkdir(exist_ok=True)
def identity(): print(f'Họ tên: {HO_TEN} | MSSV: {MSSV}')
def pick_file(keywords, extensions=('.csv','.xlsx','.xls','.txt','.html','.htm')):
    fs=[p for p in BASE.rglob('*') if p.is_file() and p.suffix.lower() in extensions and not OUTPUT in p.parents]
    for keys in keywords:
        matches=[f for f in fs if all(w in f.name.lower() for w in keys)]
        if matches:return sorted(matches,key=lambda f:len(f.name))[0]
    raise FileNotFoundError(f'Không tìm thấy tập tin chứa từ khóa {keywords} trong {BASE}. File hiện có: {[x.name for x in fs[:30]]}')
def read_table(file, **kw):
    if file.suffix.lower() in ('.xls','.xlsx'):return pd.read_excel(file,**kw)
    for enc in ['utf-8-sig','utf-8','cp1252','latin-1']:
        try:
            df=pd.read_csv(file,encoding=enc,**kw)
            if df.shape[1]==1 and ';' in str(df.columns[0]):df=pd.read_csv(file,encoding=enc,sep=';',**kw)
            return df
        except UnicodeDecodeError:continue
    return pd.read_csv(file,encoding='latin-1',**kw)
def norm_cols(df):
    df=df.copy(); df.columns=[re.sub(r'[^a-z0-9]+','',str(c).strip().lower()) for c in df.columns]
    return df
def rule_table(freq, confidence):
    if freq.empty:return pd.DataFrame()
    try:r=association_rules(freq,metric='confidence',min_threshold=confidence,num_itemsets=int(freq.attrs.get('n_transactions',0)) or None)
    except TypeError:r=association_rules(freq,metric='confidence',min_threshold=confidence)
    if len(r):
        r=r.copy()
        for c in ['antecedents','consequents']:r[c]=r[c].map(lambda s:', '.join(sorted(map(str,s))))
    return r
identity(); print('Thư mục dữ liệu:', BASE)
# Tự dò dữ liệu được Colab upload vào /content hoặc LAB1_data.
def ensure_dataset(filename):
    for d in [BASE,Path('/content'),Path('/content/LAB1_data'),Path('/mnt/data')]:
        p=d/filename
        if p.exists(): return p
    try:
        from google.colab import files
        print('Upload dataset:',filename)
        upl=files.upload()
        for name,b in upl.items(): (BASE/Path(name).name).write_bytes(b)
        if (BASE/filename).exists():return BASE/filename
    except ImportError:pass
    raise FileNotFoundError(filename)


### Chuẩn bị — Dữ liệu Online Retail

In [ ]:
retail_file=ensure_dataset('Online Retail.xlsx')
retail=norm_cols(read_table(retail_file))
retail['description']=retail['description'].astype('string').str.strip()
retail=retail.dropna(subset=['invoiceno','description']).copy()
retail['invoiceno']=retail['invoiceno'].astype(str).str.strip()
retail=retail.loc[~retail['invoiceno'].str.contains('C',case=False,na=False)].copy()
retail['quantity']=pd.to_numeric(retail['quantity'],errors='coerce')

### Chuẩn bị — Hàm khai phá

In [ ]:
def make_basket(df,country):
    sub=df.loc[df['country'].astype(str).str.casefold()==country.casefold(),['invoiceno','description','quantity']].dropna()
    if sub.empty:raise ValueError(f'Không có giao dịch từ {country}')
    hot=sub.groupby(['invoiceno','description'])['quantity'].sum().unstack(fill_value=0)
    basket=hot.gt(0)
    basket=basket.drop(columns=['POSTAGE'],errors='ignore')
    # Loại những hóa đơn không còn mặt hàng dương
    basket=basket.loc[basket.any(axis=1)]
    return hot,basket
def run_market_basket(df,country,sup,conf):
    hot,basket=make_basket(df,country)
    print(f'{country}: {len(basket)} hóa đơn, {basket.shape[1]} loại hàng, min_sup={sup:.0%}, min_conf={conf:.0%}')
    print('Hot encoding (tổng Quantity):');display(hot.iloc[:5,:8])
    print('One-hot boolean:');display(basket.iloc[:5,:8])
    fa=apriori(basket,min_support=sup,use_colnames=True,low_memory=True)
    ff=fpgrowth(basket,min_support=sup,use_colnames=True)
    ra=rule_table(fa,conf);rf=rule_table(ff,conf)
    print('Apriori:',len(fa),'tập phổ biến,',len(ra),'luật')
    print('FP-Growth:',len(ff),'tập phổ biến,',len(rf),'luật')
    display(ra.sort_values('confidence',ascending=False).head(20) if len(ra) else 'Không có luật đạt ngưỡng')
    if len(ra):
        plt.figure(figsize=(7,4));plt.scatter(ra['support'],ra['confidence'],alpha=.7);plt.xlabel('Support');plt.ylabel('Confidence');plt.title(f'Apriori: {country}');plt.show()
    # So khớp tập phổ biến không phụ thuộc thứ tự
    a={frozenset(s) for s in fa.itemsets};f={frozenset(s) for s in ff.itemsets}
    print('Hai thuật toán cùng tập phổ biến?',a==f)
    return {'hot':hot,'basket':basket,'apriori':fa,'fpgrowth':ff,'rules_apriori':ra,'rules_fpgrowth':rf}


### Kết quả bài làm

In [ ]:
identity()
result=run_market_basket(retail,'Germany',0.05,0.5)
result['rules_apriori'].to_csv(OUTPUT/'OnlineRetail_Germany_rules.csv',index=False)